# 🧠 Pipeline ML para Detecção de Alzheimer via EEG
## BrainLat Dataset — Classificação Automática AD vs. HC

---

> **Objetivo:** Classificar pacientes com Doença de Alzheimer (AD) e Controles Saudáveis (HC)  
> a partir de **features espectrais de EEG** extraídas do dataset BrainLat.

---

### 📋 Estrutura do Notebook

| Fase | Descrição |
|------|-----------|
| **0 — Configuração** | Importações globais e definição de caminhos |
| **1 — Download dos Dados** | Download via Synapse do dataset BrainLat |
| **2 — Exploração do Dataset** | Metadados demográficos e sinais EEG brutos |
| **3 — Pré-processamento & Extração** | Filtragem, épocas e features PSD (Welch) |
| **4 — Pipeline de Machine Learning** | Validação LOSO blindada por sujeito |
| **5 — Visualização dos Resultados** | ROC, matrizes de confusão e benchmarking |
| **6 — Explicabilidade (XAI)** | SHAP, Permutation Importance e erros |

---

### 📦 Dataset
- **Fonte:** [BrainLat — Synapse](https://www.synapse.org/)
- **Grupos:** AD (Alzheimer), HC (Healthy Controls)
- **Formato dos sinais:** EEGLAB `.set` / `.fdt`
- **Países:** Argentina (AR) e Chile (CL)

---


---
## ⚙️ FASE 0 — Configuração Global

Todas as importações e constantes são centralizadas aqui.  
**Execute esta fase antes de qualquer outra.**


In [ ]:
# ─────────────────────────────────────────────────────────────
# 0.1 — Importações Globais
# ─────────────────────────────────────────────────────────────
import os
import re
import warnings
import traceback
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import mne
from mne.time_frequency import psd_array_welch

from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import (
    roc_auc_score, roc_curve, auc,
    recall_score, f1_score, precision_score,
    accuracy_score, balanced_accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay,
    average_precision_score
)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from xgboost import XGBClassifier

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("⚠️  SHAP não instalado. Execute: pip install shap")

try:
    from neuroCombat import neuroCombat
    COMBAT_AVAILABLE = True
except ImportError:
    COMBAT_AVAILABLE = False

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

# Estilo global dos gráficos
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})

SEED = 42
np.random.seed(SEED)
print("✅ Importações concluídas com sucesso.")


In [ ]:
# ─────────────────────────────────────────────────────────────
# 0.2 — Configuração de Caminhos e Constantes
# ─────────────────────────────────────────────────────────────
#
# ⚠️  ADAPTE os caminhos abaixo para o seu ambiente!
#     Apenas altere esta célula — o restante do notebook
#     utilizará as variáveis aqui definidas.
# ─────────────────────────────────────────────────────────────

# Pasta raiz onde os dados serão salvos
PASTA_RAIZ       = "./Dataset_EEG_Alzheimer"
PASTA_AD         = os.path.join(PASTA_RAIZ, "dataset_eeg_alzheimer")
PASTA_HC         = os.path.join(PASTA_RAIZ, "dataset_eeg_hc")

# IDs das pastas no Synapse (não alterar)
SYNAPSE_ID_AD    = "syn53222482"
SYNAPSE_ID_HC    = "syn53222486"

# Arquivos de saída
OUT_CSV_FULL     = "eeg_features_brainlat_FULL_normalizado.csv"
OUT_CSV_FALHAS   = "eeg_features_brainlat_falhas_normalizado.csv"

# ─── Bandas de Frequência (Hz) ───────────────────────────────
BANDS = {
    "Delta": (0.5,  4.0),
    "Theta": (4.0,  8.0),
    "Alpha": (8.0, 13.0),
    "Beta" : (13.0, 30.0),
    "Gamma": (30.0, 45.0),
}
FMIN, FMAX = 0.5, 45.0

# ─── Features utilizadas no modelo ───────────────────────────
# Rel_Delta_mean foi excluída: variância zero no dataset (coluna morta)
FEATURE_COLS = [
    'Rel_Theta_mean',
    'Rel_Alpha_mean',
    'Rel_Beta_mean',
    'Rel_Gamma_mean',
    'Razao_Theta_Alpha',
    'Razao_Theta_Beta',
    'Spectral_Entropy',
]

print("✅ Configuração carregada.")
print(f"   Pasta AD  : {PASTA_AD}")
print(f"   Pasta HC  : {PASTA_HC}")
print(f"   Features  : {FEATURE_COLS}")


---
## 📥 FASE 1 — Download dos Dados (BrainLat / Synapse)

O dataset BrainLat está hospedado na plataforma [Synapse](https://www.synapse.org/).  
Para fazer o download, você precisa:

1. Criar uma conta gratuita em [synapse.org](https://www.synapse.org/)
2. Aceitar os termos de uso do dataset
3. Gerar um **Personal Access Token** em: `Profile → Personal Access Token → Create`
4. Colar o token na variável `SEU_TOKEN_AQUI` abaixo

> **Segurança:** nunca compartilhe seu token. Não o cole em repositórios públicos.

---


In [ ]:
# ─────────────────────────────────────────────────────────────
# 1.1 — Download do grupo AD (Alzheimer)
# ─────────────────────────────────────────────────────────────
import synapseclient
import synapseutils

# 🔑 SUBSTITUA o texto abaixo pelo seu token pessoal do Synapse
SEU_TOKEN = "SEU_TOKEN_AQUI"

syn = synapseclient.Synapse()
syn.login(authToken=SEU_TOKEN)

os.makedirs(PASTA_AD, exist_ok=True)
print(f"⬇️  Baixando dados do grupo AD (ID: {SYNAPSE_ID_AD})...")
synapseutils.syncFromSynapse(syn, SYNAPSE_ID_AD, path=PASTA_AD)
print("✅ Download AD concluído!")


In [ ]:
# ─────────────────────────────────────────────────────────────
# 1.2 — Download do grupo HC (Healthy Controls)
# ─────────────────────────────────────────────────────────────
os.makedirs(PASTA_HC, exist_ok=True)
print(f"⬇️  Baixando dados do grupo HC (ID: {SYNAPSE_ID_HC})...")
synapseutils.syncFromSynapse(syn, SYNAPSE_ID_HC, path=PASTA_HC)
print("✅ Download HC concluído!")


In [ ]:
# ─────────────────────────────────────────────────────────────
# 1.3 — Verificação dos Arquivos Baixados
# ─────────────────────────────────────────────────────────────
print("=" * 55)
print("  VERIFICAÇÃO DOS ARQUIVOS .set BAIXADOS")
print("=" * 55)

contagem = {"AD": [], "HC": []}

for grupo, pasta in [("AD", PASTA_AD), ("HC", PASTA_HC)]:
    for raiz, _, arquivos in os.walk(pasta):
        for arq in arquivos:
            if arq.endswith(".set"):
                caminho = os.path.join(raiz, arq)
                tamanho_mb = os.path.getsize(caminho) / (1024 ** 2)
                contagem[grupo].append((arq, tamanho_mb))

for grupo, lista in contagem.items():
    print(f"\n📂 {grupo}: {len(lista)} arquivos .set")
    for nome, tam in lista[:5]:
        print(f"   ✓ {nome}  ({tam:.1f} MB)")
    if len(lista) > 5:
        print(f"   ... e mais {len(lista)-5} arquivo(s).")

if not contagem["AD"] and not contagem["HC"]:
    print("\n❌ Nenhum arquivo .set encontrado.")
    print("   Verifique se o download foi concluído corretamente.")
else:
    print(f"\n✅ Total de sujeitos encontrados: "
          f"AD={len(contagem['AD'])} | HC={len(contagem['HC'])}")

# ── Gráfico rápido de distribuição de tamanhos ──────────────
all_sizes_ad = [t for _, t in contagem["AD"]]
all_sizes_hc = [t for _, t in contagem["HC"]]

if all_sizes_ad or all_sizes_hc:
    fig, ax = plt.subplots(figsize=(9, 4))
    if all_sizes_ad:
        ax.hist(all_sizes_ad, bins=15, alpha=0.7, color='#d62728', label='AD')
    if all_sizes_hc:
        ax.hist(all_sizes_hc, bins=15, alpha=0.7, color='#2ca02c', label='HC')
    ax.set_xlabel("Tamanho do arquivo (MB)")
    ax.set_ylabel("Número de arquivos")
    ax.set_title("Distribuição dos Tamanhos dos Arquivos EEG por Grupo")
    ax.legend()
    plt.tight_layout()
    plt.show()


---
## 🔍 FASE 2 — Exploração do Dataset

Antes de processar os sinais, é importante entender:
- Quantos participantes existem em cada grupo?
- Qual é a distribuição demográfica (idade, sexo, país)?
- Quais as características técnicas dos sinais (duração, taxa de amostragem, canais)?
- Como é o sinal EEG bruto de um sujeito típico?

---


In [ ]:
# ─────────────────────────────────────────────────────────────
# 2.1 — Análise Demográfica
# ─────────────────────────────────────────────────────────────
arq_demo_ad = os.path.join(PASTA_AD, "demographics_ad_eeg_data.csv")
arq_demo_hc = os.path.join(PASTA_HC, "demographics_hc_eeg_data.csv")  # ajuste se necessário

try:
    df_ad = pd.read_csv(arq_demo_ad)
    df_hc = pd.read_csv(arq_demo_hc)
    df_ad["grupo"] = "AD"
    df_hc["grupo"] = "HC"
    df_demo = pd.concat([df_ad, df_hc], ignore_index=True)

    print(f"✅ Carregados: {len(df_ad)} AD  |  {len(df_hc)} HC")
    print(f"   Colunas disponíveis: {list(df_demo.columns)}")
    display(df_demo.head(4))

    # ── Painel de Gráficos Demográficos ─────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle("Perfil Demográfico do Dataset BrainLat", fontsize=14, fontweight='bold')

    # Contagem por grupo
    contagens = df_demo['grupo'].value_counts()
    bars = axes[0].bar(contagens.index, contagens.values,
                       color=['#d62728', '#2ca02c'], edgecolor='white', linewidth=1.5)
    axes[0].bar_label(bars, fontsize=13, fontweight='bold', padding=4)
    axes[0].set_title("Participantes por Grupo")
    axes[0].set_ylabel("N")
    axes[0].set_ylim(0, contagens.max() * 1.2)

    # Distribuição de idades (se coluna existir)
    col_idade = next((c for c in df_demo.columns if 'age' in c.lower() or 'ida' in c.lower()), None)
    if col_idade:
        for grupo, cor in [("AD", "#d62728"), ("HC", "#2ca02c")]:
            dados = df_demo.loc[df_demo['grupo'] == grupo, col_idade].dropna()
            axes[1].hist(dados, bins=12, alpha=0.7, color=cor, label=grupo)
        axes[1].set_title("Distribuição de Idades")
        axes[1].set_xlabel("Idade (anos)")
        axes[1].set_ylabel("N")
        axes[1].legend()
    else:
        axes[1].text(0.5, 0.5, "Coluna de\nidade não encontrada",
                     ha='center', va='center', transform=axes[1].transAxes, fontsize=12)
        axes[1].set_axis_off()

    # Distribuição por sexo (se coluna existir)
    col_sexo = next((c for c in df_demo.columns if 'sex' in c.lower() or 'gen' in c.lower()), None)
    if col_sexo:
        for gi, (grupo, cor) in enumerate([("AD", "#d62728"), ("HC", "#2ca02c")]):
            dados = df_demo.loc[df_demo['grupo'] == grupo, col_sexo].value_counts()
            offset = gi * 0.35 - 0.175
            bars2 = axes[2].bar([x + offset for x in range(len(dados))],
                                dados.values, width=0.35, color=cor, alpha=0.8, label=grupo)
        axes[2].set_xticks(range(len(dados)))
        axes[2].set_xticklabels(dados.index)
        axes[2].set_title("Distribuição por Sexo")
        axes[2].set_ylabel("N")
        axes[2].legend()
    else:
        axes[2].text(0.5, 0.5, "Coluna de\nsexo não encontrada",
                     ha='center', va='center', transform=axes[2].transAxes, fontsize=12)
        axes[2].set_axis_off()

    plt.tight_layout()
    plt.show()

except FileNotFoundError as e:
    print(f"⚠️  Arquivo CSV demográfico não encontrado: {e}")
    print("    Verifique os caminhos PASTA_AD e PASTA_HC.")


In [ ]:
# ─────────────────────────────────────────────────────────────
# 2.2 — Metadados Técnicos dos Sinais EEG
# ─────────────────────────────────────────────────────────────
print("🔄 Escaneando metadados técnicos dos arquivos EEG...\n")

lista_tecnica = []

for grupo, pasta in [("AD", PASTA_AD), ("HC", PASTA_HC)]:
    arquivos = list(Path(pasta).rglob("*.set"))
    for arq in arquivos:
        nome = arq.name
        # Infere país pela estrutura de pastas
        parts_upper = {p.upper() for p in arq.parts}
        pais = "Argentina" if "AR" in parts_upper else ("Chile" if "CL" in parts_upper else "Outro")
        try:
            raw = mne.io.read_raw_eeglab(str(arq), preload=False, verbose=False)
            tipos = Counter(raw.get_channel_types())
            n_eeg = tipos.get('eeg', 0)
            lista_tecnica.append({
                "Grupo"       : grupo,
                "País"        : pais,
                "Arquivo"     : nome,
                "Duração (s)" : round(raw.times[-1], 2),
                "Fs (Hz)"     : raw.info['sfreq'],
                "N_Canais"    : raw.info['nchan'],
                "N_EEG"       : n_eeg,
                "Bad_Ch"      : len(raw.info['bads']),
            })
        except Exception as e:
            lista_tecnica.append({
                "Grupo": grupo, "País": pais, "Arquivo": nome,
                "Duração (s)": None, "Fs (Hz)": None, "N_Canais": None,
                "N_EEG": None, "Bad_Ch": None,
            })

df_tecnico = pd.DataFrame(lista_tecnica)
print(f"✅ {len(df_tecnico)} arquivos escaneados")
display(df_tecnico.head(8))

# ── Painel de Métricas Técnicas ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Características Técnicas dos Sinais EEG", fontsize=13, fontweight='bold')

# Duração dos registros
for grupo, cor in [("AD", "#d62728"), ("HC", "#2ca02c")]:
    sub = df_tecnico.loc[df_tecnico['Grupo'] == grupo, 'Duração (s)'].dropna()
    axes[0].hist(sub, bins=12, alpha=0.7, color=cor, label=grupo)
axes[0].set_title("Duração dos Registros")
axes[0].set_xlabel("Duração (s)")
axes[0].set_ylabel("N")
axes[0].legend()

# Taxa de amostragem
fs_counts = df_tecnico['Fs (Hz)'].value_counts()
axes[1].bar(fs_counts.index.astype(str), fs_counts.values, color='#1f77b4')
axes[1].set_title("Taxa de Amostragem (Fs)")
axes[1].set_xlabel("Fs (Hz)")
axes[1].set_ylabel("N de arquivos")

# Número de canais
ch_counts = df_tecnico['N_Canais'].value_counts().sort_index()
axes[2].bar(ch_counts.index.astype(str), ch_counts.values, color='#9467bd')
axes[2].set_title("Número de Canais")
axes[2].set_xlabel("N Canais")
axes[2].set_ylabel("N de arquivos")

plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────
# 2.3 — Visualização de um Sinal EEG Bruto (Inspeção Visual)
# ─────────────────────────────────────────────────────────────
# Abre o primeiro arquivo .set encontrado e plota os 10 primeiros segundos
# de 5 canais representativos para inspeção visual.

primeiro_set = next(Path(PASTA_AD).rglob("*.set"), None)
if primeiro_set is None:
    primeiro_set = next(Path(PASTA_HC).rglob("*.set"), None)

if primeiro_set:
    print(f"📂 Visualizando: {primeiro_set.name}")
    raw_viz = mne.io.read_raw_eeglab(str(primeiro_set), preload=True, verbose=False)

    sfreq   = raw_viz.info['sfreq']
    canais  = [c for c in raw_viz.ch_names if raw_viz.get_channel_types([c])[0] == 'eeg'][:5]
    t_max   = min(10.0, raw_viz.times[-1])
    n_pts   = int(t_max * sfreq)
    tempos  = raw_viz.times[:n_pts]
    dados   = raw_viz.get_data(picks=canais)[:, :n_pts]

    fig, axes = plt.subplots(len(canais), 1, figsize=(14, 7), sharex=True)
    fig.suptitle(f"EEG Bruto — {primeiro_set.name} (primeiros {t_max:.0f}s)",
                 fontsize=12, fontweight='bold')
    for i, (ch, ax) in enumerate(zip(canais, axes)):
        ax.plot(tempos, dados[i] * 1e6, lw=0.7, color=plt.cm.tab10(i))
        ax.set_ylabel(f"{ch}\n(µV)", fontsize=8)
        ax.yaxis.set_tick_params(labelsize=7)
    axes[-1].set_xlabel("Tempo (s)")
    plt.tight_layout()
    plt.show()
else:
    print("❌ Nenhum arquivo .set encontrado. Execute a Fase 1 primeiro.")


---
## 🔬 FASE 3 — Pré-processamento e Extração de Features EEG

### Pipeline de Processamento

```
.set (EEGLAB)
   ↓ Carregamento Robusto
   ↓ Re-referenciação (média global)
   ↓ Filtro Passa-Faixa (0.5 – 45 Hz)
   ↓ Seleção de Canais EEG
   ↓ Épocas Fixas (4 s)
   ↓ Normalização por Sujeito (RMS global = 1)
   ↓ PSD via Welch (por época)
   ↓ Features: Potência Relativa por Banda + Razões + Entropia Espectral
   ↓ CSV com todas as épocas de todos os sujeitos
```

### Features Extraídas

| Feature | Descrição | Relevância Clínica |
|---------|-----------|-------------------|
| `Rel_Theta_mean` | Potência relativa Theta (4–8 Hz) | ↑ em AD: slowing cortical |
| `Rel_Alpha_mean` | Potência relativa Alpha (8–13 Hz) | ↓ em AD: disfunção tálamo-cortical |
| `Rel_Beta_mean` | Potência relativa Beta (13–30 Hz) | Variável em AD |
| `Rel_Gamma_mean` | Potência relativa Gamma (30–45 Hz) | ↓ em AD: déficit de sincronização |
| `Razao_Theta_Alpha` | Theta / Alpha | Marcador clássico de declínio cognitivo |
| `Razao_Theta_Beta` | Theta / Beta | Relacionada ao slowing geral |
| `Spectral_Entropy` | Entropia espectral | ↓ em AD: regularidade patológica do EEG |

---


In [ ]:
# ─────────────────────────────────────────────────────────────
# 3.1 — Funções Auxiliares: Identificação e Carregamento
# ─────────────────────────────────────────────────────────────

def extrair_subject_id(arq_path: Path) -> str:
    """Extrai o ID do sujeito (ex.: sub-30007) do nome do arquivo."""
    m = re.search(r"(sub-\d+)", arq_path.stem, flags=re.IGNORECASE)
    if m:
        return m.group(1)
    m = re.search(r"sub[-_]?(\d+)", arq_path.stem, flags=re.IGNORECASE)
    if m:
        return f"sub-{m.group(1)}"
    return arq_path.stem


def inferir_pais(arq_path: Path) -> str:
    """Infere o país de origem pelo caminho (AR=Argentina, CL=Chile)."""
    parts = {p.upper() for p in arq_path.parts}
    if "AR" in parts: return "Argentina"
    if "CL" in parts: return "Chile"
    return "Outro"


def carregar_raw_seguro(arq_path: Path):
    """
    Carrega arquivo .set de forma robusta.
    Se o .fdt correspondente estiver ausente, tenta cópia temporária.
    """
    try:
        return mne.io.read_raw_eeglab(str(arq_path), preload=True, verbose=False)
    except Exception as e:
        msg = str(e).lower()
        if "fdt" in msg or "not found" in msg:
            fdt_path = arq_path.with_suffix(".fdt")
            if fdt_path.exists():
                from tempfile import TemporaryDirectory
                with TemporaryDirectory() as tmp:
                    tmp_p = Path(tmp)
                    (tmp_p / arq_path.name).write_bytes(arq_path.read_bytes())
                    (tmp_p / fdt_path.name).write_bytes(fdt_path.read_bytes())
                    return mne.io.read_raw_eeglab(
                        str(tmp_p / arq_path.name), preload=True, verbose=False)
        raise e

print("✅ Funções auxiliares definidas.")


In [ ]:
# ─────────────────────────────────────────────────────────────
# 3.2 — Função de Extração de Features Espectrais (PSD Welch)
# ─────────────────────────────────────────────────────────────

def extrair_metricas_eeg(data: np.ndarray, sfreq: float) -> dict:
    """
    Calcula features espectrais para UMA época EEG.

    Parâmetros
    ----------
    data  : array 2D (n_canais × n_amostras), já normalizado (RMS global ≈ 1)
    sfreq : frequência de amostragem (Hz)

    Retorna
    -------
    dict com potências relativas por banda, razões espectrais e entropia.
    """
    data = np.asarray(data, dtype=float)
    if data.ndim != 2:
        raise ValueError(f"Esperado array 2D, recebido shape={data.shape}")

    # PSD via Welch — n_fft adaptado ao comprimento da janela
    n_fft = min(512, data.shape[-1])
    psds, freqs = psd_array_welch(
        data, sfreq=sfreq, fmin=FMIN, fmax=FMAX,
        n_fft=n_fft, verbose=False
    )
    psd_media = np.mean(psds, axis=0)          # média sobre todos os canais

    # Integração via trapézio (compatível com numpy >= 1.22)
    integrar = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

    # Potência absoluta por banda
    pots = {}
    for banda, (fmin_b, fmax_b) in BANDS.items():
        idx = (freqs >= fmin_b) & (freqs < fmax_b)
        pots[banda] = float(integrar(psd_media[idx], freqs[idx])) if np.sum(idx) >= 2 else 0.0

    total = sum(pots.values()) + 1e-12   # evita divisão por zero

    # Features relativas
    f = {
        "Rel_Delta_mean": pots["Delta"] / total,   # mantida para diagnóstico (var ≈ 0)
        "Rel_Theta_mean": pots["Theta"] / total,
        "Rel_Alpha_mean": pots["Alpha"] / total,
        "Rel_Beta_mean" : pots["Beta"]  / total,
        "Rel_Gamma_mean": pots["Gamma"] / total,
    }
    # Razões espectrais — marcadores clínicos de slowing cognitivo
    f["Razao_Theta_Alpha"] = f["Rel_Theta_mean"] / (f["Rel_Alpha_mean"] + 1e-12)
    f["Razao_Theta_Beta"]  = f["Rel_Theta_mean"] / (f["Rel_Beta_mean"]  + 1e-12)

    # Entropia espectral (desordem do espectro)
    psd_norm = psd_media / (np.sum(psd_media) + 1e-12)
    f["Spectral_Entropy"] = float(-np.sum(psd_norm * np.log2(psd_norm + 1e-12)))

    return f

print("✅ Função de extração espectral definida.")
print("   Features que serão geradas:")
for k in ["Rel_Delta_mean","Rel_Theta_mean","Rel_Alpha_mean",
          "Rel_Beta_mean","Rel_Gamma_mean",
          "Razao_Theta_Alpha","Razao_Theta_Beta","Spectral_Entropy"]:
    print(f"   • {k}")


In [ ]:
# ─────────────────────────────────────────────────────────────
# 3.3 — Loop Principal: Processamento e Extração de Features
# ─────────────────────────────────────────────────────────────
#
#  Para cada arquivo .set:
#    1. Re-referencia para a média global
#    2. Aplica filtro passa-faixa (0.5 – 45 Hz)
#    3. Mantém apenas canais EEG
#    4. Cria épocas fixas de 4 s
#    5. Normaliza amplitude por sujeito (RMS global = 1)
#    6. Extrai features para cada época
# ─────────────────────────────────────────────────────────────

dados_finais = []
falhas       = []

for grupo, pasta in [("AD", PASTA_AD), ("HC", PASTA_HC)]:
    arquivos = list(Path(pasta).rglob("*.set"))
    print(f"\n{'='*55}")
    print(f"  Grupo {grupo}: {len(arquivos)} arquivo(s) encontrado(s)")
    print(f"{'='*55}")

    for arq in arquivos:
        try:
            sujeito_id = extrair_subject_id(arq)
            pais       = inferir_pais(arq)
            raw        = carregar_raw_seguro(arq)

            # 1. Re-referência para a média
            raw.set_eeg_reference('average', projection=False)
            # 2. Filtro passa-faixa
            raw.filter(FMIN, FMAX, fir_design='firwin', verbose=False)
            # 3. Apenas canais EEG
            raw.pick_types(eeg=True, eog=False, ecg=False,
                           stim=False, emg=False, misc=False)
            # 4. Épocas fixas de 4 segundos
            epochs = mne.make_fixed_length_epochs(
                raw, duration=4.0, preload=True, verbose=False)
            if len(epochs) == 0:
                raise RuntimeError("Nenhuma época gerada (sinal muito curto?).")

            data  = epochs.get_data()          # (n_epochs, n_ch, n_times)
            sfreq = float(raw.info["sfreq"])

            # 5. Normalização de amplitude por sujeito (RMS global)
            global_rms = np.sqrt(np.mean(data.reshape(-1, data.shape[-1]) ** 2))
            if global_rms < 1e-12:
                global_rms = 1.0
            data_norm = data / global_rms      # RMS global ≈ 1

            # 6. Extração por época
            for i, epoca in enumerate(data_norm):
                feat = extrair_metricas_eeg(epoca, sfreq)
                feat.update({
                    "subject_id"  : sujeito_id,
                    "label"       : grupo,
                    "Pais"        : pais,
                    "epoch_id"    : i,
                    "source_file" : arq.name,
                })
                dados_finais.append(feat)

            print(f"  ✅ {sujeito_id:<18} | épocas={len(data):3d} "
                  f"| RMS={global_rms:.4e} | País={pais}")

        except Exception as e:
            falhas.append({
                "grupo"    : grupo,
                "arquivo"  : arq.name,
                "caminho"  : str(arq),
                "erro"     : str(e),
                "traceback": traceback.format_exc(),
            })
            print(f"  ❌ {arq.name}  →  {e}")


In [ ]:
# ─────────────────────────────────────────────────────────────
# 3.4 — Salvamento e Resumo da Extração
# ─────────────────────────────────────────────────────────────
df_full   = pd.DataFrame(dados_finais)
df_falhas = pd.DataFrame(falhas)

if df_full.empty:
    raise RuntimeError(
        "Nenhuma época foi extraída. Verifique os caminhos e o download.")

# Reordena: metadados → features
cols_meta = ["subject_id", "label", "Pais", "epoch_id", "source_file"]
cols_feat = [c for c in df_full.columns if c not in cols_meta]
df_full   = df_full[cols_meta + cols_feat]

df_full.to_csv(OUT_CSV_FULL,   index=False, encoding="utf-8-sig")
df_falhas.to_csv(OUT_CSV_FALHAS, index=False, encoding="utf-8-sig")

n_subj_ad = df_full.loc[df_full['label']=='AD', 'subject_id'].nunique()
n_subj_hc = df_full.loc[df_full['label']=='HC', 'subject_id'].nunique()
n_ep_ad   = (df_full['label']=='AD').sum()
n_ep_hc   = (df_full['label']=='HC').sum()

print(f"\n{'='*55}")
print("  RESUMO DA EXTRAÇÃO")
print(f"{'='*55}")
print(f"  Sujeitos AD  : {n_subj_ad}")
print(f"  Sujeitos HC  : {n_subj_hc}")
print(f"  Épocas AD    : {n_ep_ad:,}")
print(f"  Épocas HC    : {n_ep_hc:,}")
print(f"  Total épocas : {len(df_full):,}")
print(f"  Falhas       : {len(df_falhas)}")
print(f"{'='*55}")
print(f"\n  💾 Salvo em: {OUT_CSV_FULL}")

# ── Visualização das Features Extraídas ─────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("Distribuição das Features Espectrais por Grupo", fontsize=13, fontweight='bold')

for i, feat in enumerate(cols_feat):
    ax = axes[i // 4][i % 4]
    for grupo, cor in [("AD", "#d62728"), ("HC", "#2ca02c")]:
        sub = df_full.loc[df_full['label'] == grupo, feat].dropna()
        ax.hist(sub, bins=30, alpha=0.6, color=cor, label=grupo, density=True)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel("Valor", fontsize=8)
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=8)

# Oculta subplots vazios
for j in range(len(cols_feat), 8):
    axes[j // 4][j % 4].set_visible(False)

plt.tight_layout()
plt.show()


---
## 🤖 FASE 4 — Pipeline de Machine Learning

### Estratégia de Validação: Leave-One-Subject-Out (LOSO)

O maior risco em classificadores EEG é o **vazamento de dados (data leakage)**:  
treinar e testar com épocas do mesmo sujeito infla artificialmente o desempenho.

A solução adotada é a validação **LOSO por sujeito**:

```
Para cada sujeito S:
  ├── TREINO  : todos os outros N-1 sujeitos
  └── TESTE   : todas as épocas de S (predição por média de probabilidades)
```

**Garantias contra leakage:**
- `StandardScaler` ajustado **apenas no conjunto de treino**
- Sujeito de teste nunca visto durante treinamento ou normalização
- Predição final = média das probabilidades das épocas → decisão por sujeito

---


In [ ]:
# ─────────────────────────────────────────────────────────────
# 4.1 — Carregamento do CSV e Validação de Qualidade
# ─────────────────────────────────────────────────────────────
df_audit = pd.read_csv(OUT_CSV_FULL)

# Verifica integridade
colunas_req = ["subject_id", "label"] + FEATURE_COLS
faltando = [c for c in colunas_req if c not in df_audit.columns]
if faltando:
    raise ValueError(f"Colunas ausentes no CSV: {faltando}")

# Auditoria de variância (detecta features mortas antes do treino)
print("🔍 Auditoria de variância das features:\n")
for col in FEATURE_COLS:
    std_val = df_audit[col].std()
    flag = "⚠️  VAR≈0 — EXCLUIR" if std_val < 1e-8 else "✅"
    print(f"  {flag}  {col:<28}  std={std_val:.6f}")

print(f"\n✅ CSV carregado: {df_audit.shape[0]:,} épocas × {df_audit.shape[1]} colunas")
print(f"   Sujeitos únicos : {df_audit['subject_id'].nunique()}")
print(f"   AD / HC         : {(df_audit['label']=='AD').sum()} / {(df_audit['label']=='HC').sum()}")

# ── Heatmap de correlação entre features ────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
corr = df_audit[FEATURE_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, ax=ax, annot_kws={"size": 9})
ax.set_title("Correlação entre Features Espectrais", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────
# 4.2 — Funções Auxiliares do Pipeline ML
# ─────────────────────────────────────────────────────────────

def aplicar_standard_scaler(X_tr, X_te, cols):
    """
    StandardScaler correto: estatísticas ajustadas APENAS no treino.
    Evita vazamento de informação do conjunto de teste.
    """
    scaler   = StandardScaler()
    X_tr_out = X_tr.copy()
    X_te_out = X_te.copy()
    X_tr_out[cols] = scaler.fit_transform(X_tr[cols])   # fit + transform no treino
    X_te_out[cols] = scaler.transform(X_te[cols])        # apenas transform no teste
    return X_tr_out, X_te_out


def loso_correto(X_df, y_arr, groups_arr, modelo_base,
                 feature_cols=FEATURE_COLS, verbose=True):
    """
    Validação Leave-One-Subject-Out sem vazamento de dados.

    Parâmetros
    ----------
    X_df        : pd.DataFrame (épocas × features)
    y_arr       : array binário — 0=HC, 1=AD
    groups_arr  : array com subject_id por época
    modelo_base : estimador sklearn (clonado a cada fold)
    feature_cols: lista de features utilizadas
    verbose     : imprime progresso por fold

    Retorna
    -------
    pd.DataFrame com [subject_id, real, proba, n_epocas] e métricas em .attrs
    """
    X_df       = X_df.reset_index(drop=True)
    y_arr      = np.asarray(y_arr).ravel()
    groups_arr = np.asarray(groups_arr).ravel()

    if not (len(X_df) == len(y_arr) == len(groups_arr)):
        raise ValueError("X, y e groups com tamanhos incompatíveis.")

    cols_ausentes = [c for c in feature_cols if c not in X_df.columns]
    if cols_ausentes:
        raise ValueError(f"Features ausentes: {cols_ausentes}")

    std_zero = [c for c in feature_cols if X_df[c].std() < 1e-8]
    if std_zero:
        raise ValueError(f"Features com variância zero detectadas: {std_zero}\n"
                         "Remova-as de FEATURE_COLS antes de prosseguir.")

    logo      = LeaveOneGroupOut()
    preds     = []
    skipped   = []
    n_subj    = len(np.unique(groups_arr))

    if verbose:
        print(f"   Iniciando {n_subj} folds LOSO...\n")

    for fold_i, (train_idx, test_idx) in enumerate(
            logo.split(X_df, y_arr, groups=groups_arr)):

        subj_id  = groups_arr[test_idx][0]
        X_tr_raw = X_df.iloc[train_idx]
        X_te_raw = X_df.iloc[test_idx]
        y_tr     = y_arr[train_idx]
        y_te     = y_arr[test_idx]

        if len(np.unique(y_tr)) < 2:
            skipped.append(subj_id)
            if verbose:
                print(f"  ⚠  Fold {fold_i+1:02d} [{subj_id}] — treino sem as duas classes")
            continue

        # Normalização: fit APENAS no treino
        X_tr_sc, X_te_sc = aplicar_standard_scaler(X_tr_raw, X_te_raw, feature_cols)

        # Treino com modelo clonado (estado virgem a cada fold)
        modelo = clone(modelo_base)
        modelo.fit(X_tr_sc[feature_cols], y_tr)

        # Predição por época → agregação por sujeito (média)
        proba_epocas  = modelo.predict_proba(X_te_sc[feature_cols])[:, 1]
        proba_sujeito = float(np.mean(proba_epocas))
        label_sujeito = int(y_te[0])

        preds.append({
            "subject_id": subj_id,
            "real"      : label_sujeito,
            "proba"     : proba_sujeito,
            "n_epocas"  : len(proba_epocas),
        })

        if verbose:
            acerto = "✓" if int(proba_sujeito >= 0.5) == label_sujeito else "✗"
            print(f"  {acerto}  [{fold_i+1:02d}/{n_subj}] {subj_id:<18} "
                  f"| real={'AD' if label_sujeito else 'HC'} "
                  f"| proba={proba_sujeito:.3f} | épocas={len(proba_epocas)}")

    if not preds:
        print("  ⚠  Nenhuma predição gerada.")
        return None

    if skipped and verbose:
        print(f"\n  ⚠  {len(skipped)} fold(s) ignorado(s): {skipped}")

    df_preds = pd.DataFrame(preds)
    pred_bin = (df_preds["proba"] >= 0.5).astype(int)

    df_preds.attrs["auc"]  = roc_auc_score(df_preds["real"], df_preds["proba"])
    df_preds.attrs["sens"] = recall_score(df_preds["real"], pred_bin, pos_label=1, zero_division=0)
    df_preds.attrs["spec"] = recall_score(df_preds["real"], pred_bin, pos_label=0, zero_division=0)
    df_preds.attrs["f1"]   = f1_score(df_preds["real"], pred_bin, zero_division=0)

    return df_preds

print("✅ Pipeline LOSO definido e pronto para execução.")


In [ ]:
# ─────────────────────────────────────────────────────────────
# 4.3 — Definição dos Modelos de Classificação
# ─────────────────────────────────────────────────────────────
#
#  Três modelos são comparados no benchmark LOSO:
#  - Random Forest  : robusto, interpretável via SHAP
#  - SVM (kernel RBF): funciona bem em espaços de alta dimensão
#  - XGBoost        : boosting de árvores, geralmente mais preciso
# ─────────────────────────────────────────────────────────────

modelos_corrigidos = {
    "RandomForest": RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1,
    ),
    "SVM_RBF": CalibratedClassifierCV(
        SVC(kernel="rbf", C=1.0, gamma="scale",
            class_weight="balanced", random_state=SEED),
        cv=3,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=SEED,
        n_jobs=-1,
    ),
}

print("✅ Modelos configurados:")
for nome, clf in modelos_corrigidos.items():
    print(f"   • {nome}")


In [ ]:
# ─────────────────────────────────────────────────────────────
# 4.4 — Execução do Benchmarking LOSO
# ─────────────────────────────────────────────────────────────

X_full_df = df_audit[FEATURE_COLS].copy().reset_index(drop=True)
y_full    = (df_audit['label'] == 'AD').astype(int).values
groups    = df_audit['subject_id'].values

assert len(X_full_df) == len(y_full) == len(groups),     "Tamanhos incompatíveis entre X, y e groups."

print("=" * 60)
print(f"  Dataset  : {len(groups):,} épocas  |  {len(np.unique(groups))} sujeitos")
print(f"  AD épocas: {int(y_full.sum())}  |  HC épocas: {int((1-y_full).sum())}")
print(f"  Features : {FEATURE_COLS}")
print("=" * 60 + "\n")

resultados_seguros = {}

for nome, clf in modelos_corrigidos.items():
    print(f"\n▶ Executando LOSO — {nome}")
    print("─" * 50)

    df_res = loso_correto(
        X_df         = X_full_df,
        y_arr        = y_full,
        groups_arr   = groups,
        modelo_base  = clf,
        feature_cols = FEATURE_COLS,
        verbose      = True,
    )

    if df_res is None or df_res.empty:
        print(f"  ❌ {nome}: loso_correto() retornou vazio — verifique o dataset.")
        continue

    df_res['pred_bin'] = (df_res['proba'] >= 0.5).astype(int)
    auc_v  = roc_auc_score(df_res['real'], df_res['proba'])
    sens_v = recall_score(df_res['real'], df_res['pred_bin'], pos_label=1, zero_division=0)
    spec_v = recall_score(df_res['real'], df_res['pred_bin'], pos_label=0, zero_division=0)

    resultados_seguros[nome] = {
        'auc': round(auc_v, 4), 'sens': round(sens_v, 4),
        'spec': round(spec_v, 4), 'df_preds': df_res,
    }
    print(f"\n  ✅ {nome}  |  AUC={auc_v:.4f}  Sens={sens_v:.4f}  Spec={spec_v:.4f}")
    print("─" * 50)

# ── Tabela resumo no terminal ────────────────────────────────
if resultados_seguros:
    print("\n" + "=" * 52)
    print("  RESULTADO FINAL — LOSO POR SUJEITO")
    print("=" * 52)
    print(f"  {'Modelo':<20} {'AUC':>7} {'Sens':>7} {'Spec':>7}")
    print(f"  {'-'*20} {'-'*7} {'-'*7} {'-'*7}")
    for nome, r in resultados_seguros.items():
        print(f"  {nome:<20} {r['auc']:>7.4f} {r['sens']:>7.4f} {r['spec']:>7.4f}")
    print("=" * 52)


---
## 📊 FASE 5 — Visualização dos Resultados

Avaliação completa do desempenho de cada modelo com:
- **Curvas ROC** — capacidade discriminativa geral (AUC)
- **Matrizes de Confusão** — padrão de acertos e erros por classe
- **Distribuições de Probabilidade** — separabilidade AD/HC
- **Tabela de Benchmark** — resumo comparativo
- **Gráfico de Barras** — comparação visual das métricas

---


In [ ]:
# ─────────────────────────────────────────────────────────────
# 5.1 — Curvas ROC (todos os modelos em uma figura)
# ─────────────────────────────────────────────────────────────
cores = ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd']

fig, ax = plt.subplots(figsize=(8, 6))

for i, (nome, r) in enumerate(resultados_seguros.items()):
    df    = r['df_preds']
    fpr, tpr, _ = roc_curve(df['real'], df['proba'])
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2, color=cores[i % len(cores)],
            label=f"{nome}  (AUC = {roc_auc_val:.3f})")

ax.plot([0, 1], [0, 1], 'k--', lw=1.2, label="Classificador aleatório")
ax.fill_between([0, 1], [0, 0], [1, 1], alpha=0.03, color='gray')
ax.set_xlabel("Taxa de Falsos Positivos (1 − Especificidade)", fontsize=11)
ax.set_ylabel("Taxa de Verdadeiros Positivos (Sensibilidade)", fontsize=11)
ax.set_title("Curvas ROC — Validação LOSO por Sujeito", fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────
# 5.2 — Matrizes de Confusão
# ─────────────────────────────────────────────────────────────
n_modelos = len(resultados_seguros)
fig, axes = plt.subplots(1, n_modelos, figsize=(5 * n_modelos, 4))
if n_modelos == 1:
    axes = [axes]

fig.suptitle("Matrizes de Confusão — LOSO por Sujeito",
             fontsize=13, fontweight='bold', y=1.02)

for ax, (nome, r) in zip(axes, resultados_seguros.items()):
    df     = r['df_preds']
    y_true = df['real']
    y_pred = (df['proba'] >= 0.5).astype(int)
    cm     = confusion_matrix(y_true, y_pred)

    disp = ConfusionMatrixDisplay(cm, display_labels=["HC", "AD"])
    disp.plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
    ax.set_title(f"{nome}\nAUC={r['auc']:.3f}", fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────
# 5.3 — Distribuição das Probabilidades de AD por Modelo
# ─────────────────────────────────────────────────────────────
n_modelos = len(resultados_seguros)
fig, axes = plt.subplots(1, n_modelos, figsize=(6 * n_modelos, 4), sharey=False)
if n_modelos == 1:
    axes = [axes]

fig.suptitle("Distribuição de Probabilidades Preditas por Grupo",
             fontsize=13, fontweight='bold')

for ax, (nome, r) in zip(axes, resultados_seguros.items()):
    df = r['df_preds']
    for label_val, cor, lbl in [(0, '#2ca02c', 'HC'), (1, '#d62728', 'AD')]:
        sub = df.loc[df['real'] == label_val, 'proba']
        ax.hist(sub, bins=12, alpha=0.6, color=cor, label=lbl, density=True)
    ax.axvline(0.5, color='black', linestyle='--', lw=1.5, label='Threshold 0.5')
    ax.set_title(nome, fontsize=10)
    ax.set_xlabel("Probabilidade predita de AD")
    ax.set_ylabel("Densidade")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────
# 5.4 — Tabela de Benchmark e Gráfico Comparativo
# ─────────────────────────────────────────────────────────────
rows = [
    {"Modelo": nome, "AUC": r['auc'],
     "Sensibilidade": r['sens'], "Especificidade": r['spec']}
    for nome, r in resultados_seguros.items()
]
df_benchmark = (pd.DataFrame(rows)
                .sort_values("AUC", ascending=False)
                .reset_index(drop=True))

# Formatação da tabela para exibição
df_display = df_benchmark.copy()
for col in ["AUC", "Sensibilidade", "Especificidade"]:
    df_display[col] = df_display[col].map("{:.4f}".format)

print("\n📊 Ranking de Modelos (LOSO por sujeito):")
display(df_display)

# ── Gráfico de barras agrupadas ──────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
x      = np.arange(len(df_benchmark))
width  = 0.26
metricas = ["AUC", "Sensibilidade", "Especificidade"]
cores_m  = ['#1f77b4', '#2ca02c', '#d62728']

for i, (metrica, cor) in enumerate(zip(metricas, cores_m)):
    bars = ax.bar(x + i * width, df_benchmark[metrica], width,
                  label=metrica, color=cor, alpha=0.85, edgecolor='white')
    ax.bar_label(bars, fmt='%.3f', fontsize=8, padding=2)

ax.set_xticks(x + width)
ax.set_xticklabels(df_benchmark["Modelo"], fontsize=11)
ax.set_ylabel("Score", fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_title("Benchmark dos Classificadores — LOSO por Sujeito",
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(0.5, color='gray', linestyle=':', lw=1)
plt.tight_layout()
plt.show()


---
## 🧠 FASE 6 — Explicabilidade (XAI) e Validação Neurobiológica

Esta fase responde **por que** o modelo toma suas decisões, indo além das métricas de desempenho.

### Métodos utilizados

| Técnica | O que mede |
|---------|-----------|
| **SHAP (TreeExplainer)** | Contribuição de cada feature para cada predição individual |
| **Permutation Importance** | Queda no AUC quando a feature é embaralhada (teste no conjunto de teste) |
| **Consistência do Sinal** | Proporção de folds onde a feature tem o mesmo efeito (↑/↓ AD) |
| **Análise de Erros** | Perfil espectral dos Falsos Positivos e Falsos Negativos |
| **Correlação Clínica** | Spearman entre probabilidade predita e escore cognitivo (ex.: MMSE) |
| **PDP (Partial Dependence)** | Relação marginal entre cada feature e a probabilidade de AD |

> 💡 **Hipóteses neurobiológicas esperadas:**  
> - ↑ Theta, ↑ Razão Theta/Alpha, ↑ Theta/Beta → predição AD (slowing cortical)  
> - ↓ Alpha, ↓ Spectral Entropy → predição AD (sincronização patológica)

---


In [ ]:
# ─────────────────────────────────────────────────────────────
# 6.1 — Loop LOSO com SHAP e Permutation Importance
# ─────────────────────────────────────────────────────────────
# Usa Random Forest (mais interpretável via SHAP TreeExplainer)

RANDOM_STATE     = SEED
N_REPEATS_PERM   = 10
SHAP_SAMPLE_SIZE = 200          # reduz tempo de cálculo do SHAP
CLINICAL_COLUMN  = None         # ex.: 'MMSE' se disponível em df_audit

X_full     = df_audit[FEATURE_COLS].copy().reset_index(drop=True)
y_full_xai = (df_audit['label'] == 'AD').astype(int).values
groups_xai = df_audit['subject_id'].values
n_features = len(FEATURE_COLS)

logo = LeaveOneGroupOut()

results_per_fold      = []
shap_values_list      = []
perm_importances_list = []
false_pos_subjects    = []
false_neg_subjects    = []

print("=" * 65)
print("  LOSO COM EXPLICABILIDADE — Random Forest")
print("=" * 65)

for fold_i, (train_idx, test_idx) in enumerate(
        logo.split(X_full, y_full_xai, groups_xai)):

    test_subject = groups_xai[test_idx[0]]

    X_train = X_full.iloc[train_idx]; X_test = X_full.iloc[test_idx]
    y_train = y_full_xai[train_idx];  y_test  = y_full_xai[test_idx]

    if len(np.unique(y_train)) < 2:
        continue   # fold inválido

    # ── Normalização e treino ────────────────────────────────
    X_tr_sc, X_te_sc = aplicar_standard_scaler(X_train, X_test, FEATURE_COLS)
    rf = RandomForestClassifier(n_estimators=200, max_depth=10,
                                random_state=RANDOM_STATE, n_jobs=-1)
    rf.fit(X_tr_sc[FEATURE_COLS], y_train)

    # ── Predição por sujeito ─────────────────────────────────
    proba_subj = float(np.mean(rf.predict_proba(X_te_sc[FEATURE_COLS])[:, 1]))
    pred_subj  = 1 if proba_subj >= 0.5 else 0
    y_true_subj = int(y_test[0])

    # ── SHAP (subamostrado se necessário) ────────────────────
    if SHAP_AVAILABLE:
        X_te_sample = (X_te_sc[FEATURE_COLS].iloc[
            np.random.choice(len(X_te_sc), min(SHAP_SAMPLE_SIZE, len(X_te_sc)), replace=False)]
            if len(X_te_sc) > SHAP_SAMPLE_SIZE else X_te_sc[FEATURE_COLS])
        explainer  = shap.TreeExplainer(rf)
        shap_test  = explainer.shap_values(X_te_sample)
        shap_test  = shap_test[1] if isinstance(shap_test, list) else shap_test[:, :, 1]
        shap_values_list.append(shap_test)

    # ── Permutation Importance ───────────────────────────────
    perm_imp = permutation_importance(rf, X_te_sc[FEATURE_COLS], y_test,
                                      n_repeats=N_REPEATS_PERM,
                                      scoring='roc_auc',
                                      random_state=RANDOM_STATE)
    perm_importances_list.append(perm_imp.importances_mean)

    # ── Análise de erros ─────────────────────────────────────
    if pred_subj == 1 and y_true_subj == 0:
        false_pos_subjects.append(test_subject)
    elif pred_subj == 0 and y_true_subj == 1:
        false_neg_subjects.append(test_subject)

    results_per_fold.append({
        'fold': fold_i, 'test_subject': test_subject,
        'y_true': y_true_subj, 'y_pred': pred_subj, 'proba': proba_subj,
    })

print(f"\n✅ {len(results_per_fold)} folds concluídos.")
print(f"   Falsos Positivos (HC→AD): {len(false_pos_subjects)}")
print(f"   Falsos Negativos (AD→HC): {len(false_neg_subjects)}")


In [ ]:
# ─────────────────────────────────────────────────────────────
# 6.2 — Importância Global das Features
# ─────────────────────────────────────────────────────────────

perm_all  = np.array(perm_importances_list)   # (n_folds, n_features)
perm_mean = perm_all.mean(axis=0)
perm_std  = perm_all.std(axis=0)

importance_df = pd.DataFrame({
    'Feature'               : FEATURE_COLS,
    'Permutation_ΔAUC_mean' : perm_mean,
    'Permutation_ΔAUC_std'  : perm_std,
}).sort_values('Permutation_ΔAUC_mean', ascending=False)

if SHAP_AVAILABLE and shap_values_list:
    shap_all  = np.vstack(shap_values_list)        # (total_test_samples, n_features)
    shap_mean = np.abs(shap_all).mean(axis=0)
    shap_std  = np.abs(shap_all).std(axis=0)
    importance_df['SHAP_abs_mean'] = shap_mean
    importance_df['SHAP_abs_std']  = shap_std
    # Direção dominante do efeito SHAP
    importance_df['SHAP_direction'] = np.sign(shap_all).mean(axis=0)
    importance_df = importance_df.sort_values('SHAP_abs_mean', ascending=False)

print("\n🧠 Importância das Features:")
display(importance_df.round(4))

# ── Gráfico Permutation Importance ──────────────────────────
n_cols = 2 if SHAP_AVAILABLE and shap_values_list else 1
fig, axes = plt.subplots(1, n_cols, figsize=(7 * n_cols, 5))
if n_cols == 1:
    axes = [axes]

# Permutation
top = importance_df.head(len(FEATURE_COLS))
sns.barplot(data=top, x='Permutation_ΔAUC_mean', y='Feature',
            ax=axes[0], palette='rocket_r', orient='h')
axes[0].errorbar(x=top['Permutation_ΔAUC_mean'], y=range(len(top)),
                 xerr=top['Permutation_ΔAUC_std'], fmt='none', color='black', capsize=3)
axes[0].set_title("Permutation Importance\n(queda no AUC ao permutar feature)", fontsize=11)
axes[0].set_xlabel("Δ AUC médio")
axes[0].axvline(0, color='gray', linestyle='--', lw=1)

# SHAP (se disponível)
if SHAP_AVAILABLE and shap_values_list and 'SHAP_abs_mean' in importance_df.columns:
    top_s = importance_df.sort_values('SHAP_abs_mean', ascending=False).head(len(FEATURE_COLS))
    sns.barplot(data=top_s, x='SHAP_abs_mean', y='Feature',
                ax=axes[1], palette='viridis_r', orient='h')
    axes[1].errorbar(x=top_s['SHAP_abs_mean'], y=range(len(top_s)),
                     xerr=top_s['SHAP_abs_std'], fmt='none', color='black', capsize=3)
    axes[1].set_title("Importância SHAP Global\n(|SHAP| médio ± DP)", fontsize=11)
    axes[1].set_xlabel("|SHAP value| médio")

plt.suptitle("Importância das Features — Validação LOSO", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────
# 6.3 — SHAP Summary Plot e Análise de Consistência
# ─────────────────────────────────────────────────────────────
if SHAP_AVAILABLE and shap_values_list:

    # ── SHAP Beeswarm (summary plot) ────────────────────────
    print("🌐 SHAP Summary Plot — todos os sujeitos de teste agregados:")
    shap.summary_plot(shap_all, X_full[FEATURE_COLS],
                      show=False, max_display=len(FEATURE_COLS))
    plt.title("SHAP Summary Plot (AD vs. HC)\nPontos deslocados para direita = aumenta P(AD)",
              fontsize=11)
    plt.tight_layout()
    plt.show()

    # ── Consistência do sinal SHAP entre folds ───────────────
    # Para cada feature, calcula a proporção de folds onde o sinal médio é positivo
    consistency = []
    for f_idx in range(n_features):
        signs = [np.sign(sv[:, f_idx].mean()) for sv in shap_values_list]
        pos_prop = np.mean(np.array(signs) == 1)
        consistency.append(max(pos_prop, 1 - pos_prop))

    importance_df['SHAP_consistency'] = consistency

    fig, ax = plt.subplots(figsize=(9, 6))
    sc = ax.scatter(importance_df['SHAP_abs_mean'], importance_df['SHAP_consistency'],
                    c=importance_df['Permutation_ΔAUC_mean'],
                    cmap='coolwarm', s=120, alpha=0.85, edgecolors='white', linewidth=0.8)
    for _, row in importance_df.iterrows():
        ax.annotate(row['Feature'],
                    (row['SHAP_abs_mean'], row['SHAP_consistency']),
                    xytext=(5, 5), textcoords='offset points', fontsize=9, alpha=0.8)
    ax.set_xlabel("Importância SHAP absoluta (|SHAP| médio)", fontsize=11)
    ax.set_ylabel("Consistência do sinal SHAP entre folds", fontsize=11)
    ax.set_title("Consistência Neurobiológica vs. Importância", fontsize=12, fontweight='bold')
    plt.colorbar(sc, label="Permutation ΔAUC")
    ax.set_ylim(0.4, 1.05)
    ax.axhline(0.75, color='gray', linestyle='--', lw=1, label='75% consistência')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  SHAP não disponível. Execute: pip install shap")


In [ ]:
# ─────────────────────────────────────────────────────────────
# 6.4 — Análise de Erros: Perfil Espectral de FP e FN
# ─────────────────────────────────────────────────────────────

tp_subj = [r['test_subject'] for r in results_per_fold if r['y_true']==1 and r['y_pred']==1]
tn_subj = [r['test_subject'] for r in results_per_fold if r['y_true']==0 and r['y_pred']==0]

grupos_erro = {
    'FP (HC→AD)': false_pos_subjects,
    'FN (AD→HC)': false_neg_subjects,
    'TP (AD→AD)': tp_subj,
    'TN (HC→HC)': tn_subj,
}

print("🔍 Análise de Erros:")
for nome, lista in grupos_erro.items():
    print(f"   {nome:<18}: {len(lista)} sujeito(s)")
    if lista[:5]:
        print(f"   {'':18}  {lista[:5]}")

# Heatmap com perfil espectral médio por grupo de erro
err_data = {}
for nome, lista in grupos_erro.items():
    if lista:
        mask = df_audit['subject_id'].isin(lista)
        err_data[nome] = df_audit.loc[mask, FEATURE_COLS].mean()
    else:
        err_data[nome] = pd.Series(np.nan, index=FEATURE_COLS)

if any(len(v) > 0 for v in grupos_erro.values()):
    err_df = pd.DataFrame(err_data).T.dropna(how='all')
    scaler_err = StandardScaler()
    err_df_scaled = pd.DataFrame(
        scaler_err.fit_transform(err_df),
        index=err_df.index, columns=err_df.columns)

    fig, ax = plt.subplots(figsize=(12, 4))
    sns.heatmap(err_df_scaled, cmap='RdBu_r', center=0, annot=True,
                fmt='.2f', linewidths=0.5,
                cbar_kws={'label': 'Z-score por feature'}, ax=ax)
    ax.set_title("Perfil Espectral Médio por Grupo de Erro (padronizado)",
                 fontsize=12, fontweight='bold')
    ax.set_xlabel("Feature")
    ax.set_ylabel("Grupo de Predição")
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────
# 6.5 — Partial Dependence Plots das 3 Features Mais Importantes
# ─────────────────────────────────────────────────────────────
#
# PDPs mostram como varia P(AD) ao longo do range de cada feature
# (modelo treinado em TODO o dataset — apenas para visualização).
# ─────────────────────────────────────────────────────────────

top3 = importance_df['Feature'].head(3).tolist()

rf_full = RandomForestClassifier(n_estimators=200, max_depth=10,
                                 random_state=RANDOM_STATE, n_jobs=-1)
rf_full.fit(X_full[FEATURE_COLS], y_full_xai)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Partial Dependence Plots — Top 3 Features\n"
             "(modelo treinado em todos os dados — uso ilustrativo)",
             fontsize=12, fontweight='bold')

for ax, feat in zip(axes, top3):
    PartialDependenceDisplay.from_estimator(
        rf_full, X_full[FEATURE_COLS], features=[feat],
        ax=ax, grid_resolution=30, random_state=RANDOM_STATE,
        line_kw={'color': '#1f77b4', 'lw': 2.5})
    ax.set_title(feat, fontsize=10)
    ax.set_ylabel("P(AD)", fontsize=9)
    ax.set_xlabel(feat, fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────
# 6.6 — Correlação com Dados Clínicos (MMSE ou similar)
# ─────────────────────────────────────────────────────────────
#
# Se df_audit possuir uma coluna cognitiva (ex.: 'MMSE'),
# calculamos a correlação de Spearman entre P(AD) predita e o escore.
#
# Define CLINICAL_COLUMN = 'MMSE'  (ou o nome real da coluna)
# na célula 0.2 para ativar esta análise.
# ─────────────────────────────────────────────────────────────
from scipy import stats

if CLINICAL_COLUMN and CLINICAL_COLUMN in df_audit.columns:
    clinical_subj = df_audit.groupby('subject_id')[CLINICAL_COLUMN].mean()
    proba_map     = {r['test_subject']: r['proba'] for r in results_per_fold}
    common        = set(proba_map.keys()) & set(clinical_subj.index)

    if common:
        probas_c = [proba_map[s] for s in common]
        scores_c = [clinical_subj[s] for s in common]
        rho, p   = stats.spearmanr(probas_c, scores_c)
        print(f"\n📉 Correlação de Spearman — P(AD) vs. {CLINICAL_COLUMN}:")
        print(f"   r = {rho:.3f}  |  p = {p:.4f}")
        if abs(rho) > 0.4 and p < 0.05:
            direcao = "negativa" if rho < 0 else "positiva"
            print(f"   → Correlação {direcao} significativa —")
            print("     conforme o escore cognitivo piora, P(AD) aumenta (esperado).")

        # Scatterplot
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.scatter(scores_c, probas_c, alpha=0.7, color='#1f77b4', edgecolors='white', s=60)
        m, b = np.polyfit(scores_c, probas_c, 1)
        xs = np.linspace(min(scores_c), max(scores_c), 100)
        ax.plot(xs, m * xs + b, 'r--', lw=1.8)
        ax.set_xlabel(CLINICAL_COLUMN, fontsize=11)
        ax.set_ylabel("P(AD) predita", fontsize=11)
        ax.set_title(f"P(AD) vs. {CLINICAL_COLUMN}\n"
                     f"Spearman r={rho:.3f}, p={p:.4f}", fontsize=11)
        plt.tight_layout()
        plt.show()
    else:
        print("⚠️  Não foi possível alinhar os escores clínicos com os sujeitos.")
else:
    print(f"ℹ️  CLINICAL_COLUMN não definida (valor atual: {CLINICAL_COLUMN}).")
    print("   Para ativar: defina CLINICAL_COLUMN = 'MMSE' na Fase 0 (célula 0.2).")


---
## ✅ Conclusão e Próximos Passos

### O que este pipeline garante

| Aspecto | Implementação |
|---------|--------------|
| **Sem data leakage** | LOSO por sujeito + StandardScaler fit apenas no treino |
| **Reprodutibilidade** | SEED fixo em todos os componentes aleatórios |
| **Auditoria de artefatos** | Detecção de features com variância zero antes do treino |
| **Transparência** | SHAP + Permutation Importance com barras de erro entre folds |
| **Validação clínica** | Correlação com escores cognitivos e análise de erros |

### Próximos Passos Sugeridos

1. **Aumentar o conjunto de features:** parâmetros de Hjorth (Mobilidade, Complexidade) com re-referência corrigida
2. **Análise de conectividade:** coerência entre pares de eletrodos em bandas theta/alpha
3. **Modelos baseados em Transformer:** EEGNet, ShallowConvNet adaptados para BrainLat
4. **XAI sobre Transformer:** attention maps como biomarcadores neurais
5. **Submissão ao artigo:** *Computers in Biology and Medicine*

---
*Notebook organizado e documentado para reprodutibilidade total.*
